# SLO-Aware Workload Autoscaling

Traditional autoscaling based on CPU or GPU utilization does not work well for
LLM inference. A GPU can be 90% utilized but still serving requests within SLO,
or it can be 50% utilized but badly overloaded because the KV-cache is full.

llm-d provides **inference-aware autoscaling** using metrics that actually
correlate with user-facing performance:

- **Queue depth**: How many requests are waiting to be served
- **TTFT (time-to-first-token)**: How long users wait for the first response token
- **KV-cache utilization**: How full the GPU memory is

This notebook covers two autoscaling approaches:
1. **HPA (Horizontal Pod Autoscaler)** - Standard Kubernetes scaling using
   custom EPP metrics
2. **WVA (Workload Variant Autoscaler)** - llm-d's multi-model optimizer that
   allocates GPU capacity across multiple models

In [ ]:
# Deploy the Prometheus adapter to expose EPP metrics as Kubernetes custom metrics
# This allows the HPA to read EPP metrics for scaling decisions

import os
os.environ["NAMESPACE"] = "llm-d"

# Install prometheus-adapter via Helm
# This translates Prometheus metrics into the Kubernetes custom metrics API

!helm repo add prometheus-community https://prometheus-community.github.io/helm-charts
!helm repo update

# Configure the adapter to expose EPP queue depth as a custom metric
adapter_values = """prometheus:
  url: http://prometheus.monitoring.svc:9090
rules:
  custom:
    - seriesQuery: 'epp_queue_depth{namespace="llm-d"}'
      resources:
        overrides:
          namespace: {resource: "namespace"}
          pod: {resource: "pod"}
      name:
        matches: "^(.*)"
        as: "epp_queue_depth"
      metricsQuery: 'avg(epp_queue_depth{<<.LabelMatchers>>})'
    - seriesQuery: 'epp_request_ttft_seconds{namespace="llm-d"}'
      resources:
        overrides:
          namespace: {resource: "namespace"}
          pod: {resource: "pod"}
      name:
        matches: "^(.*)"
        as: "epp_request_ttft_seconds_p99"
      metricsQuery: 'histogram_quantile(0.99, rate(epp_request_ttft_seconds_bucket{<<.LabelMatchers>>}[2m]))'
"""

with open("/tmp/prometheus-adapter-values.yaml", "w") as f:
    f.write(adapter_values)

!helm upgrade --install prometheus-adapter prometheus-community/prometheus-adapter \
    -f /tmp/prometheus-adapter-values.yaml \
    -n monitoring --create-namespace

print("\nPrometheus adapter deployed.")
print("EPP metrics are now available via the Kubernetes custom metrics API.")

In [ ]:
# Configure HPA using queue depth metric
# This HPA scales the vLLM deployment based on the average queue depth
# reported by the EPP

hpa_manifest = """apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: vllm-hpa
  namespace: llm-d
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: vllm
  minReplicas: 1
  maxReplicas: 10
  behavior:
    scaleUp:
      stabilizationWindowSeconds: 30
      policies:
        - type: Pods
          value: 2
          periodSeconds: 60
    scaleDown:
      stabilizationWindowSeconds: 300
      policies:
        - type: Pods
          value: 1
          periodSeconds: 120
  metrics:
    - type: Object
      object:
        describedObject:
          apiVersion: v1
          kind: Service
          name: epp
        metric:
          name: epp_queue_depth
        target:
          type: AverageValue
          averageValue: "5"
"""

with open("/tmp/vllm-hpa.yaml", "w") as f:
    f.write(hpa_manifest)

!kubectl apply -f /tmp/vllm-hpa.yaml

print("HPA configured:")
print("  Target:     vllm Deployment")
print("  Min:        1 replica")
print("  Max:        10 replicas")
print("  Metric:     epp_queue_depth")
print("  Target:     average queue depth of 5 per replica")
print("  Scale up:   +2 pods/min (after 30s stabilization)")
print("  Scale down: -1 pod/2min (after 5min stabilization)")

print("\n--- Current HPA status ---")
!kubectl get hpa vllm-hpa -n $NAMESPACE

## Understanding the HPA Configuration

### Why Queue Depth?

Queue depth is the most responsive scaling signal for LLM inference:

- **GPU utilization** is a lagging indicator - by the time it hits 100%, latency
  has already spiked.
- **Queue depth** is a leading indicator - it starts climbing *before* latency
  degrades, giving the autoscaler time to react.
- **TTFT** can also be used but reacts slightly slower since it measures completed
  requests, not pending ones.

### Target Average Value

The `averageValue: 5` target means the HPA tries to keep the average queue depth
at 5 requests per replica. This provides a buffer:

- **Too low** (e.g., 1): Scales up aggressively, wasting GPUs on idle replicas.
- **Too high** (e.g., 20): Allows queues to build up, degrading latency.
- **Sweet spot** (3-8): Keeps replicas busy without excessive queuing.

### Scale-Down Stabilization

The 5-minute stabilization window for scale-down prevents flapping:

1. Load spike arrives, queue depth rises, HPA scales up.
2. New replicas start and absorb the load, queue depth drops.
3. Without stabilization, HPA would immediately scale back down.
4. The stabilization window waits 5 minutes of consistently low queue depth
   before removing replicas.

GPU pods are expensive to start (model loading takes minutes), so conservative
scale-down is strongly recommended.

In [ ]:
# Generate load and watch replicas scale up
import subprocess
import time
import json

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d -o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

print(f"Gateway: {GATEWAY_IP}:8080")
print()
print("Phase 1: Check initial state")
!kubectl get hpa vllm-hpa -n llm-d
!kubectl get pods -n llm-d -l app=vllm

print("\nPhase 2: Generate sustained load (20 concurrent requests)")
print("Running load generator in background...")

# Start a load generator that sends requests in a loop
!python3 -c "
import subprocess, json, time, threading

stop_event = threading.Event()
request_count = [0]

def send_request():
    while not stop_event.is_set():
        payload = json.dumps({
            'model': 'Qwen/Qwen3-32B',
            'messages': [{'role': 'user', 'content': 'Write a paragraph about cloud computing.'}],
            'max_tokens': 200,
        })
        subprocess.run(
            ['curl', '-s', f'http://$GATEWAY_IP:8080/v1/chat/completions',
             '-H', 'Content-Type: application/json', '-d', payload],
            capture_output=True
        )
        request_count[0] += 1

# Start 20 concurrent request threads
threads = [threading.Thread(target=send_request) for _ in range(20)]
for t in threads:
    t.daemon = True
    t.start()

# Monitor for 2 minutes
for i in range(4):
    time.sleep(30)
    print(f'  [{(i+1)*30}s] Requests sent: {request_count[0]}')

stop_event.set()
print(f'  Load generation complete. Total requests: {request_count[0]}')
" &

# Check HPA status during load
for i in range(4):
    time.sleep(30)
    print(f"\n--- HPA status at t={30*(i+1)}s ---")
    !kubectl get hpa vllm-hpa -n llm-d
    !kubectl get pods -n llm-d -l app=vllm --no-headers | wc -l | xargs -I{} echo "  Replica count: {}"

print("\nThe HPA should have started scaling up replicas in response to")
print("increased queue depth from the load generator.")

In [ ]:
# Stop load and watch scale-down (takes ~5 minutes due to stabilization)
import time

print("Load generation stopped. Monitoring scale-down...")
print("(Scale-down has a 5-minute stabilization window)")
print()

for i in range(7):
    elapsed = i * 60
    print(f"--- t={elapsed}s after load stopped ---")
    !kubectl get hpa vllm-hpa -n llm-d
    !kubectl get pods -n llm-d -l app=vllm --no-headers | wc -l | xargs -I{} echo "  Replica count: {}"
    print()

    if i < 6:
        time.sleep(60)

print("After the stabilization window, the HPA gradually reduces replicas")
print("back to the minimum (1), removing 1 pod every 2 minutes.")
print()
print("For scale-to-zero, you would need KEDA or a custom controller,")
print("as the standard HPA requires minReplicas >= 1.")

## Workload Variant Autoscaler (WVA)

The HPA works well for single-model deployments. But in production, you often
serve **multiple models** on the same GPU cluster. The Workload Variant Autoscaler
(WVA) solves this multi-model optimization problem.

### The Problem

With multiple models, simple per-model HPA has issues:
- Models compete for the same GPU pool
- One model scaling up may force another to scale down
- No global view of capacity allocation

### The WVA Solution

The WVA treats the GPU cluster as a shared resource and optimizes allocation
across all models simultaneously:

1. Collects demand signals from all InferencePools
2. Solves an optimization problem: maximize served demand while respecting
   GPU capacity and per-model SLOs
3. Outputs replica counts for each model
4. Applies the scaling decisions via the Kubernetes API

This is especially useful for:
- **Multi-model serving** (e.g., small model for simple queries, large model
  for complex ones)
- **Model migration** (gradually shifting traffic from old to new model version)
- **Cost optimization** (packing models efficiently across GPU nodes)

In [ ]:
# Show WVA configuration for multi-model optimization
# This configures the WVA to manage two models sharing a GPU cluster

wva_config = """apiVersion: llm-d.ai/v1alpha1
kind: WorkloadVariantAutoscaler
metadata:
  name: multi-model-wva
  namespace: llm-d
spec:
  # Total GPU capacity available for all models
  totalGPUs: 16

  # Scaling interval
  evaluationInterval: 30s

  # Models to manage
  variants:
    - name: qwen3-32b
      inferencePool: qwen3-32b-pool
      gpusPerReplica: 1
      minReplicas: 1
      maxReplicas: 8
      targetMetric:
        name: epp_queue_depth
        targetValue: 5
      priority: 100  # Higher priority model gets GPUs first

    - name: llama-70b
      inferencePool: llama-70b-pool
      gpusPerReplica: 4  # TP=4 for 70B model
      minReplicas: 1
      maxReplicas: 3
      targetMetric:
        name: epp_queue_depth
        targetValue: 3
      priority: 80

  # Optimization strategy
  strategy:
    type: demand-proportional  # Allocate GPUs proportional to demand
    # Other options: priority-first, equal-share
"""

print("WVA Configuration:")
print(wva_config)

with open("/tmp/multi-model-wva.yaml", "w") as f:
    f.write(wva_config)

# Apply the WVA configuration
!kubectl apply -f /tmp/multi-model-wva.yaml

print("\nWVA deployed. It will now manage GPU allocation across both models.")
print("\nExample scenario with 16 GPUs total:")
print("  Low demand:  qwen3-32b=1 (1 GPU), llama-70b=1 (4 GPUs) = 5 GPUs used")
print("  High demand: qwen3-32b=8 (8 GPUs), llama-70b=2 (8 GPUs) = 16 GPUs used")
print("  Spike on qwen: qwen3-32b=8 (8 GPUs), llama-70b=1 (4 GPUs) = 12 GPUs used")
print("  (WVA shifts capacity to where demand is highest)")

## When to Use HPA vs WVA

| Scenario | Recommended | Why |
|----------|-------------|-----|
| Single model, single pool | **HPA** | Simple, well-understood, built into K8s |
| Single model, P/D disaggregation | **HPA** (one per pool) | Separate HPAs for prefill and decode pools |
| Multiple models, shared GPUs | **WVA** | Global optimization across models |
| Model A/B testing | **WVA** | Smoothly shift capacity between versions |
| Cost-sensitive multi-tenant | **WVA** | Pack models efficiently, respect priorities |

### Key Takeaways

1. **Never scale on GPU utilization alone.** Use queue depth or TTFT from the EPP.
2. **Set conservative scale-down windows.** GPU pods take minutes to start,
   so premature scale-down causes latency spikes when load returns.
3. **Start with HPA.** Only move to WVA when you have multiple models competing
   for the same GPU pool.
4. **Monitor the metrics.** Set up Grafana dashboards showing queue depth,
   replica count, and TTFT side by side. This makes scaling behavior visible
   and debuggable.
5. **Test with realistic load patterns.** Synthetic benchmarks often produce
   smooth traffic, but production traffic is bursty. Use recorded traffic
   replays for capacity planning.